# Improving earthquake metadata for GNSS time series analysis using Okada-based co-seismic deformation modelling
# ***Verification of Finite Source Model with IGS***
*** 
***

## **0. Workflow**

Our aim is to replicate the following:

**Earthquake 1: Turkey**
>  IGS Solution: \
https://lists.igs.org/pipermail/igsstation/2023/007685.html

> Source Parameters: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2023&mo=02&day=6&oyr=1976&omo=1&oday=1&jyr=&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.8&umw=7.8&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

**Earthquake 2: Philippines**
> IGS Solution Parameters: \
https://lists.igs.org/pipermail/igsstation/2026/008316.html

> Source: \
https://www.globalcmt.org/cgi-bin/globalcmt-cgi-bin/CMT5/form?itype=ymd&yr=2026&mo=06&day=7&oyr=1976&omo=1&oday=1&jyr=&jday=1&ojyr=1976&ojday=1&otype=nd&nday=1&lmw=7.8&umw=7.8&lms=0&ums=10&lmb=0&umb=10&llat=-90&ulat=90&llon=-180&ulon=180&lhd=0&uhd=1000&lts=-9999&uts=9999&lpe1=0&upe1=90&lpe2=0&upe2=90&list=0

## **1. Okada Functions**

In [1]:
import numpy as np

### 1.1. Okada's Coordinate System

In [2]:
def coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, U):
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, stalat, stalon)
    dEast = distance_m*np.sin(azimuth*np.pi/180.0) #m
    dNorth = distance_m*np.cos(azimuth*np.pi/180.0) #m

    x = dEast * np.sin(strike*np.pi/180.0) + dNorth * np.cos(strike*np.pi/180.0) #m
    y = -dEast * np.cos(strike*np.pi/180.0) + dNorth * np.sin(strike*np.pi/180.0) #m
    d = abs(evdep) #m
    delta = dip*np.pi/180.0 #radians
    
    p = y*np.cos(delta)+d*np.sin(delta) #m
    q = y*np.sin(delta)-d*np.cos(delta) #m

    mu = 10**9*mu #convert GPa to N/m^2
    lam = 10**9*lam #convert GPa to N/m^2
    return(
        U*np.cos(rake*np.pi/180.0), #m #U1
        U*np.sin(rake*np.pi/180.0), #m #U2
        x, #m #x
        y, #m #y
        d, #m #d
        mu, #GPa->N/m^2 #mu
        lam, #GPa->N/m^2 #lam
        delta, #rad #delta
        p, #m #p
        q #m #q
    )

### 1.2. Parameters needed for Chinnery Notation

In [3]:
def chinnery (x, p, q, L, W, delta):
        xi = [x+L/2,x-L/2] #m #xi
        eta = [p+W/2,p-W/2] #m #eta

        ytil = [0,0]
        dtil = [0,0]
        R = [[0,0],[0,0]]
        X = [0,0]

        for i in range(2):
            ytil[i]=eta[i]*np.cos(delta)+q*np.sin(delta) #m #ytil
            dtil[i]=eta[i]*np.sin(delta)-q*np.cos(delta) #m #dtil
            X[i]=np.sqrt(xi[i]**2+q**2) #m #X
            for j in range(2):
                R[i][j]=np.sqrt(xi[i]**2.0+eta[j]**2.0+q**2.0) #m #R

        return(xi, eta, ytil, dtil, R, X)

### 1.3. Displacement Functions and Intermediate Variables

In [4]:
def disp_strike (xi, eta, q, ytil, dtil, R, I1, I2, I4, delta, U1):
    if(abs(q)<1e-14):
        atanterm=0
    else:
        atanterm=np.arctan2((xi*eta),(q*R))
    if (abs(R+eta)<1e-14):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
    -U1/(2.0*np.pi)*(((xi*q)/R*Retaterm)+atanterm+I1*np.sin(delta)),
    -U1/(2.0*np.pi)*(((ytil*q)/R*Retaterm)+(q*np.cos(delta))*Retaterm+I2*np.sin(delta)),
    -U1/(2.0*np.pi)*(((dtil*q)/R*Retaterm)+(q*np.sin(delta))*Retaterm+I4*np.sin(delta))
    )

In [5]:
def disp_dip (xi, eta, q, ytil, dtil, R, I1, I3, I5, delta, U2):
    if(abs(q)<1e-14):
        atanterm=0
    else:
        atanterm=np.arctan2((xi*eta),(q*R))
    return (
    -U2/(2.0*np.pi)*(((q/R)-I3*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((ytil*q)/(R*(R+xi))+np.cos(delta)*atanterm-I1*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((dtil*q)/(R*(R+xi))+np.sin(delta)*atanterm-I5*np.sin(delta)*np.cos(delta)))
    )

In [6]:
def disp_I (xi, eta, q, ytil, dtil, R, X, lam, mu, delta):
    if (abs(R+eta)<1e-14):
        lnterm = -np.log(R-eta)
    else:
        lnterm = np.log(R+eta)
    if (abs(np.cos(delta))<1e-14):
        I3 = mu/(2*(lam+mu))*(eta/(R+dtil)+(ytil*q)/(R+dtil)**2-lnterm)
        I2_1stterm =mu/(lam+mu)*(-lnterm)-I3
        return(
            -mu/(2*(lam+mu))*xi*q/(R+dtil)**2,
            I2_1stterm,
            I3,
            mu/(lam+mu)*q/(R+dtil),
            mu/(lam+mu)*xi*np.sin(delta)/(R+dtil)
        )
    else:
        I4 = mu/(lam+mu)/np.cos(delta)*(np.log(R+dtil)-np.sin(delta)*lnterm)
        I3 = mu/(lam+mu)*(1/np.cos(delta)*ytil/(R+dtil)-lnterm)+np.sin(delta)/np.cos(delta)*I4
        I5 = 0 if abs(xi)<1e-14 else mu/(lam+mu)*2.0/np.cos(delta)*np.arctan2((eta*(X+q*np.cos(delta))+X*(R+X)*np.sin(delta)),(xi*(R+X)*np.cos(delta)))
        I2_1stterm =mu/(lam+mu)*(-lnterm)-I3
        return(
            mu/(lam+mu)*(-1)/np.cos(delta)*xi/(R+dtil)-np.sin(delta)/(np.cos(delta))*I5,
            I2_1stterm,
            I3,
            I4, 
            I5
        )

In [7]:
def displacement (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, U, L, W) :
    U1, U2, x, y, d, mu, lam, delta, p, q = coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, U)
    xi, eta, ytil, dtil, R, X = chinnery (x, p, q, L, W, delta)
    ux, uy, uz = [], [], []
    for i in range(2):
        for j in range (2):
            I1, I2, I3, I4, I5 = disp_I (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], X[i], lam, mu, delta)
            ux_strike, uy_strike, uz_strike = disp_strike (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I2, I4, delta, U1)
            ux_dip, uy_dip, uz_dip = disp_dip (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I3, I5, delta, U2)
            ux.append(ux_strike + ux_dip)
            uy.append(uy_strike + uy_dip)
            uz.append(uz_strike + uz_dip)
    return ( #chinnery
        ux[0]-ux[1]-ux[2]+ux[3], #ux
        uy[0]-uy[1]-uy[2]+uy[3], #uy
        uz[0]-uz[1]-uz[2]+uz[3] #uz
    )

### 1.4. Scaling Laws

In [8]:
def scaling (mag, m0):
    if (mag<7.6): #Yen & Ma
        if (m0<=10.0**20.0): #N-m
            return(
                10.0**(0.5*np.log10(m0)-8.08)*1000.0, #m #L
                10.0**(0.5*np.log10(m0)-8.08)*1000.0, #m #W
                10.0**1.68/100 #m #U
            )
        else:
            return(
                10.0**(1.0/3.0*np.log10(m0)-4.84)*1000.0, #m #L
                10.0**(1.0/3.0*np.log10(m0)-5.27)*1000.0, #m #W
                10.0**(1.0/3.0*np.log10(m0)-4.37)/100 #m #U
            )
    else: #Mai & Beroza
        return(
            10.0**(0.39*np.log10(m0)-6.13)*1000.0, #m #L
            10.0**(0.32*np.log10(m0)-5.05)*1000.0, #m #W
            10.0**(0.29*np.log10(m0)-3.34)/100 #m #U
        )

### 1.5. Rotation to LonLat Coordinates

In [9]:
def NE_rotate(ux, uy, uz, strike):
    return(
        (ux * np.cos(strike*np.pi/180.0) + uy * np.sin(strike*np.pi/180.0))*10.0**3.0, #mm #dN
        (ux * np.sin(strike*np.pi/180.0) - uy * np.cos(strike*np.pi/180.0))*10.0**3.0, #mm #dE
        uz*10.0**3.0 #mm #dH
    )

## **2. Earthquake 1: Turkey**

In [10]:
from obspy import read_events
from obspy import UTCDateTime
from obspy.geodetics import gps2dist_azimuth
import pandas as pd

### 2.1. Earthquake Parameters

In [41]:
event_time = "2023-02-06 01:17:36"

strike1, dip1, rake1 = 51.0, 70.0, -4.0 #Nodal Plane 1
strike2, dip2, rake2 = 143.0, 86.0, -160.0 #Nodal Plane 2

evlat = 37.170 #degN
evlon = 37.030 #degE
evdep = 15.1*1000 #km to m

mag = 7.8
m0 = 5.8e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

### 2.2. Station Parameters

In [42]:
sta_ids = ['ANK2', 'ARUC', 'BSHM', 'DRAG', 'DYNG', 'HAMD', 'ISBA', 'ISTA', 'IZMI', 'KHAR', 'KRS1', 'MERS', 'MIKL', 'NICO', 'ORID', 'POLV', 'RAMO', 'SOFI', 'TEHN', 'TUBI', 'ZECK']
sta_lats = [39.843, 40.286, 32.779, 31.593, 38.079, 34.869, 33.341, 41.104, 38.395, 50.005, 40.588, 36.566, 46.973, 35.141, 41.127, 49.603, 30.598, 42.556, 35.697, 40.787, 43.788] #degN
sta_lons = [32.775, 44.086, 35.020, 35.392, 23.932, 48.534, 44.438, 29.019, 27.082, 36.239, 43.093, 34.356, 31.973, 33.396, 20.794, 34.543, 34.763, 23.395, 51.334, 29.451, 41.565] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

21
21
21


### 2.3. Calculating Surface Deformation

In [43]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

In [44]:
for i in range(len(sta_ids)): # For every station

    #Compute the slip (U), length (L), and width of rupture (W) using scaling laws
    L, W, U = scaling(mag, m0)

    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/Turkey_M{mag}_{event_time}.csv", index=False)

## **3. Earthquake 2: Philippines**

### 3.1. Earthquake Parameters

In [45]:
event_time = "2026-06-07 23:37:42"

strike1, dip1, rake1 = 165.0, 44.0, 84.0 #Nodal Plane 1
strike2, dip2, rake2 = 353.0, 46.0, 96.0 #Nodal Plane 2

evlat = 5.590 #degN
evlon = 125.050 #degE
evdep = 44.0*1000 #km to m

mag = 7.8
m0 = 6.13e+27*1e-7 #Dyne-cm to N-m

mu = 30.0 #GPa #mu
lam = 30.0 #GPa #lambda

### 3.2. Station Parameters

In [46]:
sta_ids = ['BTNG', 'PGEN', 'PPPC']
sta_lats = [1.439, 6.065, 9.773] #degN
sta_lons = [125.190, 125.132, 118.740] #degE

print(len(sta_ids))
print(len(sta_lats))
print(len(sta_lons))

3
3
3


### 3.3. Calculating Surface Deformation

In [47]:
# Distances in km
dist = []

# Okada 1985 Displacements in m
# Nodal Plane 1
sta_dE1 = []
sta_dN1 = []
sta_dH1 = []

# Nodal Plane 2
sta_dE2 = []
sta_dN2 = []
sta_dH2 = []

In [48]:
for i in range(len(sta_ids)): # For every station

    #Compute the slip (U), length (L), and width of rupture (W) using scaling laws
    L, W, U = scaling(mag, m0)

    # Compute Distance
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, sta_lats[i], sta_lons[i])
    dist.append(round(distance_m/1000.0)) #km
    
    # Compute the Total Displacement for Nodal Plane 1
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike1, dip1, rake1, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike1)
    
    # Save values   
    # IGS conventions
    sta_dE1.append(round(dE,1))
    sta_dN1.append(round(dN,1))
    sta_dH1.append(round(dH,1))

    # Compute the Total Displacement for Nodal Plane 2
    ux, uy, uz = displacement (sta_lons[i], sta_lats[i], evlon, evlat, evdep, strike2, dip2, rake2, mu, lam, U, L, W)
    total_disp = np.sqrt(ux**2+uy**2+uz**2)
    dN, dE, dH = NE_rotate(ux, uy, uz, strike2)
    
    # Save values   
    # IGS conventions
    sta_dE2.append(round(dE,1))
    sta_dN2.append(round(dN,1))
    sta_dH2.append(round(dH,1))
    
# Save in dataframe
relevant_stations = pd.DataFrame()

relevant_stations["Station_ID"] = sta_ids
relevant_stations["Station Longitude (°E)"] = sta_lons
relevant_stations["Station Latitude (°N)"] = sta_lats

relevant_stations["Event Date"] = [event_time]*len(sta_ids)
relevant_stations["Event Mag"] = [mag]*len(sta_ids)
relevant_stations["Event Longitude (°E)"] = [evlon]*len(sta_ids)
relevant_stations["Event Latitude (°N)"] = [evlat]*len(sta_ids)
relevant_stations["Event Depth (km)"] = [evdep/1000.0]*len(sta_ids)

relevant_stations["Distance (km)"] = dist

relevant_stations["dE1 [mm]"] = sta_dE1
relevant_stations["dN1 [mm]"] = sta_dN1
relevant_stations["dH1 [mm]"] = sta_dH1

relevant_stations["dE2 [mm]"] = sta_dE2
relevant_stations["dN2 [mm]"] = sta_dN2
relevant_stations["dH2 [mm]"] = sta_dH2

relevant_stations.to_csv(f"IGSverify/Philippines_M{mag}_{event_time}.csv", index=False)